In [6]:
from langchain_chroma import Chroma
from langchain_aws import BedrockEmbeddings
from langchain_core.documents import Document
import json
import os
from dotenv import load_dotenv


load_dotenv(override=True)
# ----------------------------
# 1. Setup Embeddings & VectorStore
# ----------------------------
embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")

# initialize Chroma in-memory (persist_directory can be set to save to disk)
vectorstore = Chroma(
    collection_name="doc_chunks",
    embedding_function=embeddings,
    persist_directory="./chroma_db"  # change or remove if you want only in-memory
)


# ----------------------------
# 2. Sample Data: key-value with bounding box metadata
# ----------------------------
texts_with_metadata = [
    {
        "key": "Policy Number",
        "value": "6F7G8H9I0J",
        "bbox": {"x1": 100, "y1": 200, "x2": 300, "y2": 230}
    },
    {
        "key": "Insured Name",
        "value": "John Doe",
        "bbox": {"x1": 120, "y1": 250, "x2": 400, "y2": 280}
    },
    {
        "key": "Premium Amount",
        "value": "$500",
        "bbox": {"x1": 150, "y1": 300, "x2": 250, "y2": 330}
    }
]

# Convert into LangChain Documents

# Store as: "key: value" in text, and bbox as metadata
documents = []
for item in texts_with_metadata:
    text = f"{item['key']}: {item['value']}"
    metadata = {"bbox": json.dumps(item["bbox"]), "key": item["key"]}
    documents.append(Document(page_content=text, metadata=metadata))

# Add to vectorstore
vectorstore.add_documents(documents)
print("Documents added to vectorstore.")
# ----------------------------
# 3. Create Retriever
# ----------------------------
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# ----------------------------
# 4. Test Retrieval
# ----------------------------
query = "What is the policy number?"
results = retriever.get_relevant_documents(query)

print("Query:", query)
print("\nTop Matches:")
for r in results:
    print("-", r.page_content, "| Metadata:", r.metadata)


: 